### 1. Configuração do Catálogo e Dimensão Principal (DIM_MOVIES)
Nesta fase inicial da camada Gold, garantimos o isolamento da modelagem analítica ao criar a base de dados `gold` dentro do Unity Catalog (`cinedata_analytics`).
A tabela `dim_movies` é materializada utilizando a função `row_number()` através de uma *Window Function* ordenada pelo ID natural do filme. Este processo gera a *Surrogate Key* (chave substituta sequencial `sk_movie_id`), essencial para isolar a estrutura analítica das alterações nos sistemas de origem e otimizar o desempenho do modelo dimensional.

In [0]:
from pyspark.sql.functions import col, row_number, count, round, avg
from pyspark.sql.window import Window

# 1. Configuração inicial
spark.sql("USE CATALOG cinedata_analytics")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# ==============================================================================
# 2. DIM_MOVIES
# ==============================================================================
df_info = spark.read.table("silver.tb_info_filmes")
window_sk = Window.orderBy("id_filme")

df_dim_movies = df_info.withColumn("sk_movie_id", row_number().over(window_sk)) \
                       .select("sk_movie_id", "id_filme", "titulo", "data_lancamento", 
                               "ano_lancamento", "duracao_minutos", "idioma_original", 
                               "status_filme", "sinopse")

df_dim_movies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_movies")
print("gold.dim_movies criada!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_movies criada!


### 2. Dimensão de Géneros (DIM_GENRES)
Criação do catálogo unificado de géneros cinematográficos. O script extrai exclusivamente os valores únicos (`distinct()`) provenientes da tabela desnormalizada da camada Silver. A geração da chave substituta `sk_genre_id` assegura que cada género possua um identificador numérico otimizado, preparando a estrutura para futuros cruzamentos (*joins*) sem dependência de chaves textuais longas.

In [0]:
# ==============================================================================
# 3. DIM_GENRES
# ==============================================================================
df_generos_silver = spark.read.table("silver.tb_generos")
df_generos_unicos = df_generos_silver.select("nome_genero").distinct()

df_dim_genres = df_generos_unicos.withColumn("sk_genre_id", row_number().over(Window.orderBy("nome_genero"))) \
                                 .select("sk_genre_id", "nome_genero")

df_dim_genres.write.format("delta").mode("overwrite").saveAsTable("gold.dim_genres")
print("gold.dim_genres criada!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_genres criada!


### 3. Dimensões de Entidades: Pessoas e Produtoras
Separação lógica da base consolidada de entidades da camada Silver em duas dimensões distintas, focadas no grão correto de negócio:
* **DIM_COMPANIES:** Isolamento dos registos classificados como 'Produtora', gerando a sua respetiva `sk_company_id`.
* **DIM_PEOPLE:** Filtragem exclusória para capturar atores, diretores e roteiristas, aplicando o `row_number()` particionado para garantir a rastreabilidade estrutural (`sk_person_id`).
Ambas as tabelas são gravadas em formato Delta com substituição total (`overwrite`) a cada execução do *pipeline*.

In [0]:
# ==============================================================================
# 4. DIM_PEOPLE e DIM_COMPANIES
# ==============================================================================
df_pessoas_empresas = spark.read.table("silver.tb_pessoas_empresas")

# Dimensão de Pessoas (Filtra 'Produtora' de fora)
df_pessoas = df_pessoas_empresas.filter(col("tipo_entidade") != "Produtora") \
                                .select(col("nome_entidade").alias("nome_pessoa"), 
                                        col("tipo_entidade").alias("tipo_pessoa")).distinct()

df_dim_people = df_pessoas.withColumn("sk_person_id", row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa"))) \
                          .select("sk_person_id", "nome_pessoa", "tipo_pessoa")

df_dim_people.write.format("delta").mode("overwrite").saveAsTable("gold.dim_people")
print("gold.dim_people criada!")

# Dimensão de Empresas (Filtra apenas 'Produtora')
df_empresas = df_pessoas_empresas.filter(col("tipo_entidade") == "Produtora") \
                                 .select(col("nome_entidade").alias("nome_produtora")).distinct()

df_dim_companies = df_empresas.withColumn("sk_company_id", row_number().over(Window.orderBy("nome_produtora"))) \
                              .select("sk_company_id", "nome_produtora")

df_dim_companies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_companies")
print("gold.dim_companies criada!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_people criada!
gold.dim_companies criada!


### 4. Agregação Analítica: DIM_REVIEWS
Transformação dos registos granulares de avaliações individuais numa métrica consolidada de negócio. A operação agrupa os dados pelo ID do filme para calcular a volumetria absoluta (contagem de avaliações) e a métrica de qualidade (média das notas, com arredondamento exato a duas casas decimais). O cruzamento posterior com a dimensão de filmes assegura a injeção correta da *Surrogate Key*, mantendo a integridade referencial do modelo.

In [0]:
# ==============================================================================
# 5. DIM_REVIEWS (Agregada)
# ==============================================================================
df_reviews_silver = spark.read.table("silver.tb_avaliacoes_usuarios")

# Agrupar por filme, contar avaliações e calcular média arredondada
df_reviews_agg = df_reviews_silver.groupBy("id_filme") \
                                  .agg(count("nome_usuario").alias("qtd_avaliacoes_usuarios"),
                                       round(avg("nota_usuario"), 2).alias("nota_media_usuarios"))

# Cruzar com dim_movies para mapear o id natural (id_filme) para a chave substituta (sk_movie_id)
df_dim_reviews_joined = df_reviews_agg.join(df_dim_movies.select("id_filme", "sk_movie_id"), "id_filme", "inner")

df_dim_reviews = df_dim_reviews_joined.withColumn("sk_review_id", row_number().over(Window.orderBy("sk_movie_id"))) \
                                      .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")

df_dim_reviews.write.format("delta").mode("overwrite").saveAsTable("gold.dim_reviews")
print("gold.dim_reviews criada!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_reviews criada!


### 5. Tabela Fato: FACT_MOVIES_PERFORMANCE
Construção da tabela central do *Star Schema*, responsável por centralizar o histórico de transações numéricas (financeiras e de engajamento). 
* **Conversão de Tipos (Casting):** Aplicação de tipagem rigorosa, convertendo moedas para `decimal(18,2)`. Isto previne erros clássicos de arredondamento de vírgula flutuante quando a tabela for consumida por ferramentas de Business Intelligence (BI).
* **Unicidade e Relacionamento:** Consolidação das métricas num único grão (um registo por filme) e associação direta à `sk_movie_id`, formando a estrela da nossa modelagem.

In [0]:
from pyspark.sql.functions import col

# ==============================================================================
# 6. Tabela Fato: FACT_MOVIES_PERFORMANCE
# ==============================================================================
# Mude apenas a leitura destas duas bases no início da célula:
df_fin = spark.read.table("silver.tb_financeiro_filmes") # Já deduplicado no passo anterior
df_metrics = spark.read.table("silver.tb_metricas_engajamento").dropDuplicates(["id_filme"]) # Correção aqui!

# Ler dim_movies para obter a surrogate key (sk_movie_id) que fará a ligação
df_dim_movies = spark.read.table("gold.dim_movies").select("sk_movie_id", "id_filme")

# Cruzar as bases Silver para unificar as métricas no mesmo grão (1 linha por filme)
df_fact_temp = df_dim_movies.join(df_fin, "id_filme", "left") \
                            .join(df_metrics, "id_filme", "left")

# Selecionar e tipar estritamente as colunas conforme o dicionário de dados do escopo
df_fact_performance = df_fact_temp.select(
    col("sk_movie_id"),
    col("orcamento_usd").cast("decimal(18,2)"),
    col("receita_usd").cast("decimal(18,2)"),
    col("lucro_usd").cast("decimal(18,2)"),
    col("orcamento_brl").cast("decimal(18,2)"),
    col("receita_brl").cast("decimal(18,2)"),
    col("lucro_brl").cast("decimal(18,2)"),
    col("popularidade").cast("double"),
    col("nota_media_tmdb").cast("double"),
    col("qtd_votos_tmdb").cast("int"),
    col("nota_media_imdb").cast("double"),
    col("qtd_votos_imdb").cast("int")
)

df_fact_performance.write.format("delta").mode("overwrite").saveAsTable("gold.fact_movies_performance")
print("gold.fact_movies_performance criada com sucesso!")

gold.fact_movies_performance criada com sucesso!


### 6. Resolução de Relacionamentos: Tabelas-Ponte (Bridge Tables)
Para contornar o desafio de relacionamentos muitos-para-muitos (M:N) sem inflacionar a volumetria da Tabela Fato (o fenómeno de *Join Explosion*), são implementadas as tabelas-ponte. Através de *Inner Joins* rigorosos entre as chaves naturais e as novas chaves substitutas, criamos as relações entre os filmes e as dimensões periféricas (`bridge_movie_genre`, `bridge_movie_person` e `bridge_movie_company`), garantindo um modelo analítico escalável.

In [0]:
# ==============================================================================
# 7. Bridge Tables (Tabelas-Ponte)
# ==============================================================================
# 7.1 Bridge Genre
df_silver_generos = spark.read.table("silver.tb_generos")
df_dim_genres = spark.read.table("gold.dim_genres")

df_bridge_genre = df_dim_movies.join(df_silver_generos, "id_filme", "inner") \
                               .join(df_dim_genres, "nome_genero", "inner") \
                               .select("sk_movie_id", "sk_genre_id")

df_bridge_genre.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_genre")
print("gold.bridge_movie_genre criada!")

# 7.2 Preparação para Person e Company
df_silver_pessoas_empresas = spark.read.table("silver.tb_pessoas_empresas")
df_base_entidades = df_dim_movies.join(df_silver_pessoas_empresas, "id_filme", "inner")

# 7.3 Bridge Person
df_dim_people = spark.read.table("gold.dim_people")
df_bridge_person = df_base_entidades.filter(col("tipo_entidade") != "Produtora") \
                                    .join(df_dim_people, 
                                          (df_base_entidades.nome_entidade == df_dim_people.nome_pessoa) & 
                                          (df_base_entidades.tipo_entidade == df_dim_people.tipo_pessoa), 
                                          "inner") \
                                    .select("sk_movie_id", "sk_person_id")

df_bridge_person.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_person")
print("gold.bridge_movie_person criada!")

# 7.4 Bridge Company
df_dim_companies = spark.read.table("gold.dim_companies")
df_bridge_company = df_base_entidades.filter(col("tipo_entidade") == "Produtora") \
                                     .join(df_dim_companies, 
                                           df_base_entidades.nome_entidade == df_dim_companies.nome_produtora, 
                                           "inner") \
                                     .select("sk_movie_id", "sk_company_id")

df_bridge_company.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_company")
print("gold.bridge_movie_company criada!")

gold.bridge_movie_genre criada!
gold.bridge_movie_person criada!
gold.bridge_movie_company criada!


### 7. Integração com IA: Documento de Contexto (Vector Search)
Engenharia do documento não estruturado destinado ao consumo do assistente virtual (LLM).
* **Agrupamento de Arrays:** Uso de `collect_list` e `concat_ws` para unificar múltiplos diretores e atores numa única *string* legível.
* **Resiliência a Nulos (A "Casca de Banana"):** Aplicação massiva da função `coalesce()` antes de qualquer operação de `concat()`. Isto impede que um único campo vazio (como um orçamento não informado ou uma sinopse em branco) anule silenciosamente a construção do texto inteiro, garantindo que o filme não desapareça do Banco de Dados Vetorial.

In [0]:
from pyspark.sql.functions import col, concat, lit, coalesce, collect_list, concat_ws

# 1. Ler as tabelas necessárias da camada Gold
df_movies = spark.read.table("gold.dim_movies")
df_fato = spark.read.table("gold.fact_movies_performance")
df_bridge_person = spark.read.table("gold.bridge_movie_person")
df_people = spark.read.table("gold.dim_people")

# 2. Agregar os Diretores (1 linha por filme, diretores separados por vírgula)
df_diretores = df_bridge_person.join(df_people, "sk_person_id") \
    .filter(col("tipo_pessoa") == "Diretor") \
    .groupBy("sk_movie_id") \
    .agg(concat_ws(", ", collect_list("nome_pessoa")).alias("diretores"))

# 3. Agregar os Atores (1 linha por filme, atores separados por vírgula)
df_atores = df_bridge_person.join(df_people, "sk_person_id") \
    .filter(col("tipo_pessoa") == "Ator") \
    .groupBy("sk_movie_id") \
    .agg(concat_ws(", ", collect_list("nome_pessoa")).alias("atores"))

# 4. Cruzar a base principal com a Fato e as agregações de pessoas
df_context_join = df_movies.join(df_fato, "sk_movie_id", "left") \
                           .join(df_diretores, "sk_movie_id", "left") \
                           .join(df_atores, "sk_movie_id", "left")

# 5. Montar o documento LLM lidando com os Nulos (A casca de banana!)
# O coalesce() garante que, se o dado não existir, entra o texto padrão e o concat() não falha.
df_genai_context = df_context_join.select(
    col("id_filme").alias("movie_id"),
    col("titulo").alias("title"),
    concat(
        lit("O filme "), coalesce(col("titulo"), lit("Sem Título")),
        lit(", lançado no ano de "), coalesce(col("ano_lancamento").cast("string"), lit("Não Informado")),
        lit(", faturou US$ "), coalesce(col("receita_usd").cast("string"), lit("Não Informado")),
        lit(" e teve um custo de US$ "), coalesce(col("orcamento_usd").cast("string"), lit("Não Informado")),
        lit(". Estrelado por "), coalesce(col("atores"), lit("elenco não informado")),
        lit(" e dirigido por "), coalesce(col("diretores"), lit("diretor não informado")),
        lit(", o filme possui a seguinte sinopse: "), coalesce(col("sinopse"), lit("Sinopse indisponível."))
    ).alias("llm_context_document")
)

# 6. Gravar a tabela de contexto
df_genai_context.write.format("delta").mode("overwrite").saveAsTable("gold.gold_genai_movies_context")
print("gold.gold_genai_movies_context criada com sucesso para a equipa de IA!")

gold.gold_genai_movies_context criada com sucesso para a equipa de IA!
